In [11]:
import os
import gc
import zarr
import yaml
import json
import numba
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [12]:
anngeno_path = '/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag'
eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'
# olink_path = '/home/dnanexus/data_dir/olink/protrider_lite_output/log2fc.csv'
olink_path = "/home/dnanexus/data_dir/olink/olink_corrected_rint_90_pcs.parquet"

In [13]:
sample_ids = zarr.open(f'{anngeno_path}/zarr_store/samples', mode='r')[:]
var_ids = pl.read_parquet(f'{anngeno_path}/variant_metadata.parquet', columns=['id'])['id'].to_numpy()
geno = zarr.open(f'{anngeno_path}/zarr_store/genotypes', mode='r')

eur_samples = pl.read_csv(eur_samples_path).rename({'eid': 'individual'}).with_columns(
    pl.col("individual").cast(pl.Utf8)
)['individual'].to_list()

/home/dnanexus/deeprvat2-env/lib/python3.11/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)


In [22]:
olink_df = pl.read_parquet(olink_path).rename({'sample': 'individual'}).with_columns(
    pl.col("individual").cast(pl.Utf8)
).filter(
    pl.col("individual").is_in(eur_samples)
).fill_nan(None)

unique_phenotypes = [pheno for pheno in olink_df.columns if pheno != 'individual']

olink_melt = (
    olink_df.unpivot(
        index=['individual'],
        on=unique_phenotypes,
        variable_name='phenotype',
        value_name='pheno_value',
    )
    .with_columns(
        phenotype = pl.col('phenotype') + '_olink'
    )
)

olink_melt

individual,phenotype,pheno_value
str,str,f64
"""1000107""","""ENSG00000120053_olink""",-0.592022
"""1001193""","""ENSG00000120053_olink""",0.061036
"""1001694""","""ENSG00000120053_olink""",0.228713
"""1001715""","""ENSG00000120053_olink""",0.674837
"""1002486""","""ENSG00000120053_olink""",1.153924
…,…,…
"""6019565""","""ENSG00000066468_olink""",0.72708
"""6019855""","""ENSG00000066468_olink""",-0.327369
"""6021528""","""ENSG00000066468_olink""",0.991073


In [ ]:

olink_indices

Built a lookup dictionary for faster searching.
Found 40398 matching IDs.


array([    17,     23,     45, ..., 490517, 490532, 490535],
      shape=(40398,))

In [16]:
sample_ids[olink_indices]

array(['5102203', '3741733', '4072771', ..., '5141119', '5145210',
       '5733954'], shape=(40398,), dtype=StringDType())

In [17]:
len(set(olink_lazy['individual'].to_list()).intersection(set(sample_ids[olink_indices])))

40398

In [18]:
@numba.njit(parallel=True, fastmath=True)
def _fast_clip_and_sum_allels(arr):
    # Get the shape of the input array
    # Using specific dimensions for clarity with this problem
    n_samples, n_variants, _ = arr.shape
    
    # The sum of two positive int8s can be up to 254. 
    # An int16 is a safe and fast output type.
    output = np.empty((n_samples, n_variants), dtype=np.int8)
    
    # Numba's prange enables automatic parallelization across all your CPU cores
    for i in numba.prange(n_samples):
        for j in range(n_variants):
            # Read two values, perform logic, write one value.
            # This is the "fused" operation.
            val1 = arr[i, j, 0]
            val2 = arr[i, j, 1]
            
            s = 0
            # Since input is int8, this check is faster than max(0, val)
            if val1 > 0:
                s += val1
            if val2 > 0:
                s += val2
            
            output[i, j] = s
            
    return output

def process_genotype_chunk(
    geno: np.array, 
    var_ids: np.array, 
    sample_list: np.array,
    melted_pheno_df: pl.LazyFrame,
    homozygous: bool = False,
    debug: bool = False,
) -> pl.LazyFrame:
    """
    Extract genotypes for a specific gene and return as lazy DataFrame
    """
    geno_clipped = _fast_clip_and_sum_allels(geno)
    # Find heterozygous genotypes (genotype == 1)
    rows, cols = np.where(geno_clipped == 1)
    geno_melt = pl.DataFrame({
        'id': var_ids[rows],
        'individual': sample_list[cols],
        'genotype': 1
    })
    
    # Find homozygous genotypes (genotype == 2)
    if homozygous:
        rows, cols = np.where(geno_clipped == 2)
        hom = pl.DataFrame({
            'id': var_ids[rows],
            'individual': sample_list[cols],
            'genotype': 2
        })
        geno_melt = pl.concat([geno_melt, hom])

    var_pheno_df = geno_melt.lazy().join(melted_pheno_df, on='individual', how='left')
    if debug:
        # Return intermediate dataframe if debugging
        return var_pheno_df
    
    var_pheno_df = var_pheno_df.group_by(
           ['id', 'phenotype']
           ).agg([
                pl.len().alias('n_individuals'),
                pl.col('pheno_value').mean().cast(pl.Float32).alias('mean_pheno_value'),
                pl.col('pheno_value').std().cast(pl.Float32).alias('std_pheno_value'),
            ]).drop_nulls(subset=['mean_pheno_value'])
    return var_pheno_df

In [21]:
output_dir = "/home/dnanexus/data_dir/var_pheno_EUR_chunks"
chunk_size = 10_000

for chunk_num in tqdm(range(var_ids.shape[0]//chunk_size + 1)):
    tmp = process_genotype_chunk(
        geno=geno[chunk_num*chunk_size:(chunk_num+1)*chunk_size, olink_indices],
        var_ids=var_ids[chunk_num*chunk_size:(chunk_num+1)*chunk_size],
        sample_list=sample_ids,
        melted_pheno_df=olink_melt.lazy(),
        homozygous=False,
    )
    break

tmp.collect()

  0%|          | 0/183 [00:08<?, ?it/s]


id,phenotype,n_individuals,mean_pheno_value,std_pheno_value
str,str,u64,f32,f32
"""chr1:2388281:C:T""","""ENSG00000139540_olink""",1,0.700152,null
"""chr1:2303402:T:G""","""ENSG00000116147_olink""",1,1.265663,null
"""chr1:2372512:C:G""","""ENSG00000118762_olink""",1,0.530112,null
"""chr1:2304068:G:A""","""ENSG00000171988_olink""",3,-0.978153,1.093151
"""chr1:2228865:C:G""","""ENSG00000151553_olink""",170,0.042883,1.015105
…,…,…,…,…
"""chr1:11231009:G:A""","""ENSG00000161944_olink""",1,-0.076929,null
"""chr1:3731038:C:G""","""ENSG00000109758_olink""",1,1.330868,null
"""chr1:6206584:G:A""","""ENSG00000134352_olink""",4,-0.058453,0.941945
